In [ ]:
# !pip install torch -q
# !pip install peft -q
# !pip install transformers -q
# !pip install unsloth -q

In [ ]:
import torch
import gc
torch.cuda.empty_cache()
gc.collect()

In [ ]:
!unzip -q outputs.zip -d .

In [ ]:
from huggingface_hub import login
login(token="***")

In [ ]:
# %%bash
# export HF_HOME=~/hf-cache
# export HF_HUB_ENABLE_HF_TRANSFER=1  # دانلود موازی سریع
# huggingface-cli download unsloth/gemma-3-4b-it \
#   --local-dir ~/models/gemma3_4b_it_bf16 \
#   --local-dir-use-symlinks False

In [1]:
import torch
from peft import PeftModel
from transformers import AutoProcessor
from unsloth import FastVisionModel
import os

/home/user01/rahnema/load/load-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipping import of cpp extensions due to incompatible torch version 2.8.0+cu128 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info
/tmp/ipykernel_49011/4055549487.py:4: UserWarning: WARNING: Unsloth should be imported before transformers, peft to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastVisionModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
pwd

'/home/user01/rahnema/load'

In [3]:
BASE_MODEL = "../../models/gemma3_4b_it_bf16"
ADAPTER_DIR = "outputs/checkpoint-1"
MERGED_DIR = "outputs/merged_gemma3_4b_it_lora"

In [4]:
if not os.path.isdir(BASE_MODEL):
    print(f"Error: Base model not found at {BASE_MODEL}")
    print("Please run the 'download_base_model.sh' script first or correct the path.")

In [5]:
processor = AutoProcessor.from_pretrained(BASE_MODEL, trust_remote_code=True)

In [6]:
base16, _ = FastVisionModel.from_pretrained(
    BASE_MODEL,
    load_in_4bit=False,
    torch_dtype=torch.bfloat16,
    device_map="cpu",
    trust_remote_code=True,
)

Unsloth: WARNING `trust_remote_code` is True.
Are you certain you want to do remote code execution?
==((====))==  Unsloth 2025.11.1: Fast Gemma3 patching. Transformers: 4.57.1.
   \\   /|    NVIDIA GeForce RTX 4090. Num GPUs = 1. Max memory: 23.516 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


`torch_dtype` is deprecated! Use `dtype` instead!


Unsloth: Gemma3 does not support SDPA - switching to fast eager.
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  7.71it/s]


In [7]:
model16 = PeftModel.from_pretrained(base16, ADAPTER_DIR, is_trainable=False)
merged  = model16.merge_and_unload()

In [8]:
# 3) ذخیره مدل ادغام‌شده + پردازشگر
os.makedirs(MERGED_DIR, exist_ok=True)
merged.save_pretrained(MERGED_DIR, safe_serialization=True)
processor.save_pretrained(MERGED_DIR)

print(f"Saved merged model to: {MERGED_DIR}")

Saved merged model to: outputs/merged_gemma3_4b_it_lora
